# Assignment 10

### Imports

In [1]:
import os
import torch
import torch.nn as nn
import numpy as np
import json
import sys
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sys.path.append("../../MainProject/Assignment9")
from assignment9_functions import *

device = torch.device("cpu")
random_seed = 42
np.random.seed(random_seed)
torch.manual_seed(random_seed)

## Evaluate Mediapipe and our NN against kinect data

### Function for aligning mediapipe files with kinect files where frames have been excluded

In [2]:
def load_aligned_pair(mp_file, kinect_file, mp_dir, kinect_dir):
    df_mp = pd.read_csv(os.path.join(mp_dir, mp_file))
    df_kinect = pd.read_csv(os.path.join(kinect_dir, kinect_file))

    df_kinect.columns = df_kinect.columns.str.strip()

    # Align frames
    frames = df_kinect["FrameNo"].values
    df_mp = df_mp[df_mp["FrameNo"].isin(frames)]

    df_mp = df_mp.reset_index(drop=True)
    df_kinect = df_kinect.reset_index(drop=True)

    return df_mp, df_kinect

### Load kinect data and mediapipe data

In [5]:
# Load kinect data
kinect_data = "../../MainProject/Assignment9/data/kinect_good_preprocessed"
train_files_kinect, test_files_kinect = split_csvfiles(kinect_data, random_seed, 0.9, 0)


# Load Mediapipe data, ensure split is consistent with how kinect was splitted
mp_data = "../../MainProject/Assignment10/data/csv_of_all_videos"
train_files_mp = [f.replace("kinect", "mediapipe") for f in train_files_kinect]
test_files_mp = [f.replace("kinect", "mediapipe") for f in test_files_kinect]


# Align mediapipe and kinect training data
train_mp_list = []
train_kinect_list = []
for mp_file, kinect_file in zip(train_files_mp, train_files_kinect):
    df_mp, df_kinect = load_aligned_pair(mp_file, kinect_file, mp_data, kinect_data)

    train_mp_list.append(df_mp)
    train_kinect_list.append(df_kinect)

train_data_mp = pd.concat(train_mp_list, ignore_index=True)
train_data_kinect = pd.concat(train_kinect_list, ignore_index=True)

# Align mediapipe and kinect testing data
test_mp_list = []
test_kinect_list = []
for mp_file, kinect_file in zip(test_files_mp, test_files_kinect):
    df_mp, df_kinect = load_aligned_pair(mp_file, kinect_file, mp_data, kinect_data)

    test_mp_list.append(df_mp)
    test_kinect_list.append(df_kinect)

test_data_mp = pd.concat(test_mp_list, ignore_index=True)
test_data_kinect = pd.concat(test_kinect_list, ignore_index=True)

# Split based on input (x,y) and target (z)
x_train_kinect, y_train_kinect = input_target_split(train_data_kinect)
x_test_kinect, y_test_kinect = input_target_split(test_data_kinect)

x_train_mp, y_train_mp = input_target_split(train_data_mp)
x_test_mp, y_test_mp = input_target_split(test_data_mp)

# Compute scale mp -> kinect
mp_train = torch.tensor(y_train_mp.values, dtype=torch.float32).reshape(-1)
kinect_train = torch.tensor(y_train_kinect.values, dtype=torch.float32).reshape(-1)

# Compute least squares scaling factors for linear regression
a = torch.sum((mp_train - mp_train.mean()) * (kinect_train - kinect_train.mean())) / torch.sum((mp_train - mp_train.mean())**2)
b = kinect_train.mean() - a * mp_train.mean()

y_test_mp_tensor = torch.tensor(y_test_mp.values, dtype=torch.float32).to(device)
Y_test_mp_scaled = a * y_test_mp_tensor + b


# Check such that data aligns and we've succesfully excluded extra frames in mp data
print("MP train rows:", len(x_train_mp))
print("Kinect train rows:", len(y_train_kinect))

print("MP test rows:", len(x_test_mp))
print("Kinect test rows:", len(y_test_kinect))

MP train rows: 21520
Kinect train rows: 21520
MP test rows: 2485
Kinect test rows: 2485


### Train champion model on raw mp data

In [6]:
X_train_mp = torch.tensor(x_train_mp.values, dtype=torch.float32).to(device)
Y_train_mp = torch.tensor(y_train_mp.values, dtype=torch.float32).to(device)
X_test_mp = torch.tensor(x_test_mp.values, dtype=torch.float32).to(device)
Y_test_mp = torch.tensor(y_test_mp.values, dtype=torch.float32).to(device)

X_train_kinect = torch.tensor(x_train_kinect.values, dtype=torch.float32).to(device)
Y_train_kinect = torch.tensor(y_train_kinect.values, dtype=torch.float32).to(device)
X_test_kinect = torch.tensor(x_test_kinect.values, dtype=torch.float32).to(device)
Y_test_kinect = torch.tensor(y_test_kinect.values, dtype=torch.float32).to(device)


# Load Champion model
model_root = "../z_pred_models"
metadata_path = os.path.join(model_root, "metadata", "champion_info.json")
model_path = os.path.join(model_root, "champion", "champion_model.pt")

# Load Champion model
with open(metadata_path, "r") as f:
    champion_info = json.load(f)

best_config = champion_info["hyperparameters"]

best_model = build_model(best_config, device)


# Train model on scaled mediapipe data to predict kinect data
loss_fn = nn.MSELoss()

result = train_one_model(
    best_model,
    best_config,
    X_train_mp,
    Y_train_kinect,
    X_train_mp,
    Y_train_kinect,
)
best_model.load_state_dict(result["best_state"])


# Evaluate NN on raw mediapipe (x, y) against corresponding kinetic z-data
test_loss, test_metrics, test_predictions = evaluate_model(
    best_model,
    X_test_mp,
    Y_test_kinect,
    loss_fn
)

mse_nn = test_metrics['mse']
mae_nn = test_metrics['mae']
r2_nn = test_metrics['r2']
bias_nn = test_metrics['bias']


# Evaluate Mediapipes raw z-pred against corresponding kinetic z-data
mse_mp = torch.mean((Y_test_mp - Y_test_kinect)**2)
mae_mp = torch.mean(torch.abs(Y_test_mp - Y_test_kinect))
bias_mp = torch.mean(Y_test_mp - Y_test_kinect)

ss_res = torch.sum((Y_test_kinect - Y_test_mp)**2)
ss_tot = torch.sum((Y_test_kinect - torch.mean(Y_test_kinect)) **2)
r2_mp = 1- ss_res/ss_tot

# Evaluate Mediapipes scaled z-pred against corresponding kinetic z-data
mse_mp_scaled = torch.mean((Y_test_mp_scaled - Y_test_kinect)**2)
mae_mp_scaled = torch.mean(torch.abs(Y_test_mp_scaled - Y_test_kinect))
bias_mp_scaled = torch.mean(Y_test_mp_scaled - Y_test_kinect)

ss_res_scaled = torch.sum((Y_test_kinect - Y_test_mp_scaled)**2)
ss_tot_scaled = ss_tot
r2_mp_scaled = 1- ss_res_scaled/ss_tot_scaled


print("Evaluation on z-coordinate kinect data for MediaPipe and NN (trained on MediaPipe data)")
metrics_table = pd.DataFrame({
    "Model": ["Neural Network", "MediaPipe (raw)", "MediaPipe (scaled)"],
    "Loss(MSE)": [mse_nn, mse_mp.item(), mse_mp_scaled.item()],
    "MAE (m)": [mae_nn, mae_mp.item(), mae_mp_scaled.item()],
    "Bias": [bias_nn, bias_mp.item(), bias_mp_scaled.item()],
    "R2": [r2_nn, r2_mp.item(), r2_mp_scaled.item()]
})

print(metrics_table.to_string(index=False))

Evaluation on z-coordinate kinect data for MediaPipe and NN (trained on MediaPipe data)
             Model  Loss(MSE)  MAE (m)      Bias       R2
    Neural Network   0.005919 0.060450  0.026108 0.437443
   MediaPipe (raw)   0.021903 0.119302 -0.003130 0.053764
MediaPipe (scaled)   0.018557 0.101226  0.039792 0.198293
